# Mi primera tabla Gold: cobertura de viajes
Una fila conserva un viaje declarado del snapshot.
Empiezo con mode=validate; publish escribe Gold y su auditoría.


In [ ]:
# BLOQUE 1 — Configuración de cobertura Gold
from pathlib import Path
import json
import sys

dbutils.widgets.text("repo_root", "", "Raíz del repositorio en Workspace")
root_parameter = dbutils.widgets.get("repo_root").strip()
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_gold/trip_coverage.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root; falta gtfs_gold.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from gtfs_gold.trip_coverage import run_coverage

defaults = {"catalog": "workspace", "schema": "gtfs_bogota", "feed_version": "2026-09-29",
            "snapshot_sha256": "08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097", "mode": "validate"}
for key, default in defaults.items():
    dbutils.widgets.text(key, default)
parameters = {key: dbutils.widgets.get(key).strip() for key in defaults}
print(f"Cobertura Gold; snapshot: {parameters['feed_version']}; modo: {parameters['mode']}")


In [ ]:
# BLOQUE 2 — Reconciliación y ejecución explícita
# Run all vuelve a leer los widgets; validate no escribe tablas ni auditorías Gold.
result = run_coverage(spark, catalog=parameters["catalog"], schema=parameters["schema"],
                      feed_version=parameters["feed_version"], sha256=parameters["snapshot_sha256"], mode=parameters["mode"])
print(json.dumps(result, ensure_ascii=False, indent=2))
if result["persisted"]:
    print("Publicación Gold comprobada para este snapshot.")
else:
    print("Validación Gold completada; no escribí tablas ni auditorías Gold.")
